# Revisión y validación del matching YouTube ↔ Local

Objetivo: revisar los 41 matches automáticos, corregir los falsos positivos,
y recuperar posibles matches perdidos entre los 26 SIN MATCH.

Flujo:
1. Ver la tabla de matches OK ordenada por score (peores primero)
2. Editar el dict `corrections` con lo que querés cambiar
3. Opcionalmente recuperar SIN MATCH con threshold más bajo
4. Validar con OCR (disponible después del Paso 3a)
5. Guardar el CSV corregido

## 1. Setup

In [ ]:
import sys, re, warnings, unicodedata
sys.path.insert(0, '../scripts')
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt
import yaml
from pathlib import Path
from IPython.display import display

with open('../config.yaml') as f:
    config = yaml.safe_load(f)

CSV_PATH    = Path('../data/catalog/yt_vs_local_matching.csv')
OCR_REPORT  = Path('../data/catalog/subs_ocr_report.csv')
CATALOG_JSON = Path('../data/catalog/channel_catalog.json')

df = pd.read_csv(CSV_PATH)
ok  = df[df['match'] == 'OK'].copy().sort_values('score')           # peores primero
sin = df[df['match'] == 'SIN MATCH'].copy().sort_values('yt_dur_s')

print(f'Matches OK:   {len(ok)}')
print(f'Sin match:    {len(sin)}')
print(f'Total videos: {len(df)}')

## 2. Matches OK — peores primero

Revisá especialmente los de score < 0.55 (fila superior de la tabla).

In [ ]:
cols = ['yt_title', 'local_file', 'yt_dur_s', 'local_dur_s',
        'dur_score', 'kw_score', 'score', 'has_sub', 'playlist']

display(
    ok[cols]
    .style
    .background_gradient(subset=['score'], cmap='RdYlGn', vmin=0.35, vmax=0.80)
    .background_gradient(subset=['dur_score'], cmap='Blues', vmin=0.7, vmax=1.0)
    .format({
        'dur_score':   '{:.3f}',
        'kw_score':    '{:.3f}',
        'score':       '{:.3f}',
        'yt_dur_s':    '{:.0f}s',
        'local_dur_s': '{:.1f}s',
    })
    .set_properties(**{'text-align': 'left', 'font-size': '12px'})
)

## 3. Correcciones manuales

Editá el dict según lo que encontraste en la tabla:
- `"RECHAZAR"` → el match es incorrecto, no hay video local para este YT
- `"DSC_XXXX nombre.MOV"` → reemplazar por el archivo local correcto
- Para agregar un SIN MATCH recuperado: usá `rescued` más abajo

In [ ]:
# ── Editá este dict ────────────────────────────────────────────────────────────
corrections = {
    # Ejemplos:
    # "hHqyvAOZpwk": "RECHAZAR",
    # "wSd8jKV42Y8": "DSC_0873 p.87 nombre correcto.MOV",
}

# Videos SIN MATCH que querés rescatar manualmente:
rescued = {
    # "yt_id": "DSC_XXXX nombre.MOV",
}
# ──────────────────────────────────────────────────────────────────────────────

print(f'Correcciones: {len(corrections)}  |  Rescatados: {len(rescued)}')

In [ ]:
df_edit = df.copy()

for yt_id, val in corrections.items():
    if val == 'RECHAZAR':
        df_edit.loc[df_edit['yt_id'] == yt_id, ['match', 'local_file']] = ['RECHAZADO', None]
        print(f'  RECHAZADO : {yt_id}')
    else:
        df_edit.loc[df_edit['yt_id'] == yt_id, 'local_file'] = val
        print(f'  CORREGIDO : {yt_id}  →  {val}')

for yt_id, local_file in rescued.items():
    mask = df_edit['yt_id'] == yt_id
    df_edit.loc[mask, ['match', 'local_file']] = ['OK', local_file]
    print(f'  RESCATADO : {yt_id}  →  {local_file}')

print(f"\nMatches OK tras correcciones: {(df_edit['match'] == 'OK').sum()}")

## 4. Videos SIN MATCH

Lista de los 26 videos YouTube sin match local. Si reconocés alguno, agregalo al dict `rescued` arriba.

In [ ]:
display(
    sin[['yt_id', 'yt_title', 'yt_dur_s', 'playlist']]
    .style
    .set_properties(**{'text-align': 'left', 'font-size': '12px'})
    .format({'yt_dur_s': '{:.0f}s'})
)

## 5. Recuperar SIN MATCH con threshold más bajo

Re-corre el algoritmo de matching con `SCORE_THRESHOLD = 0.25` (vs 0.35 original).
Muestra candidatos nuevos que quedaron afuera del match automático.

> Requiere acceso a los MOV locales en `config.paths.videos`.

In [ ]:
import json
from analyze_raw import scan_folder
from parse_subs_docx import parse_subs_docx

RAW_DIR  = Path(config['paths']['videos'])
DOCX     = Path(config['paths']['subs_docx'])

if not RAW_DIR.exists():
    print(f'[SKIP] Carpeta de MOVs no accesible: {RAW_DIR}')
    print('  Esta celda requiere acceso al directorio Windows vía /mnt/c/')
else:
    print('Escaneando MOVs...')
    local_metas = [m for m in scan_folder(RAW_DIR) if m.readable]

    subs = parse_subs_docx(DOCX)
    sub_by_num = {s.video_num: s.text for s in subs}

    with open(CATALOG_JSON) as f:
        catalog = json.load(f)
    yt_videos = [
        {**v, 'playlist_title': pl['playlist_title']}
        for pl in catalog['playlists']
        for v in pl['videos']
    ]

    STOP = {
        'de','la','el','en','y','a','que','es','los','las','se','con','para',
        'un','una','por','del','al','como','si','lsa','guia','informacion',
        'puede','pueden','tiene','tienen','hay','ser','podes','esta','esto',
        'persona','personas','discapacidad','ciudad','buenos','aires',
    }

    def _tok(text):
        text = ''.join(c for c in unicodedata.normalize('NFD', text.lower())
                       if unicodedata.category(c) != 'Mn')
        return {t for t in re.findall(r'[a-z]+', text)
                if t not in STOP and len(t) > 3}

    local_tokens = {}
    for lm in local_metas:
        toks = _tok(lm.filename)
        if lm.video_num and lm.video_num in sub_by_num:
            toks |= _tok(sub_by_num[lm.video_num])
        local_tokens[lm.filename] = toks

    already_matched = set(df_edit.loc[df_edit['match'] == 'OK', 'local_file'].dropna())

    NEW_THRESHOLD = 0.25
    DUR_THRESHOLD = 0.70
    candidates_low = []

    for yt in yt_videos:
        if yt['video_id'] in df_edit.loc[df_edit['match'] == 'OK', 'yt_id'].values:
            continue   # ya tiene match
        best_s, best_d, best_k, best_lm = 0, 0, 0, None
        for lm in local_metas:
            if lm.filename in already_matched:
                continue
            d = min(yt.get('duration_sec', 0), lm.duration_s) / max(yt.get('duration_sec', 1), lm.duration_s, 1)
            if d < DUR_THRESHOLD:
                continue
            a = _tok(yt['title'])
            k = len(a & local_tokens[lm.filename]) / len(a | local_tokens[lm.filename]) if (a | local_tokens[lm.filename]) else 0
            s = round(0.55 * d + 0.45 * k, 3)
            if s > best_s:
                best_s, best_d, best_k, best_lm = s, d, k, lm
        if best_lm and best_s >= NEW_THRESHOLD:
            candidates_low.append({
                'yt_id': yt['video_id'], 'yt_title': yt['title'],
                'yt_dur_s': yt.get('duration_sec'),
                'local_file': best_lm.filename, 'local_dur_s': best_lm.duration_s,
                'dur_score': round(best_d, 3), 'kw_score': round(best_k, 3), 'score': best_s,
            })

    if candidates_low:
        print(f'Candidatos nuevos (threshold {NEW_THRESHOLD}):')
        display(pd.DataFrame(candidates_low).sort_values('score', ascending=False)
                .style.background_gradient(subset=['score'], cmap='RdYlGn', vmin=0.25, vmax=0.55)
                .format({'dur_score': '{:.3f}', 'kw_score': '{:.3f}', 'score': '{:.3f}',
                         'yt_dur_s': '{:.0f}s', 'local_dur_s': '{:.1f}s'}))
        print('Si alguno es correcto, agregalo al dict `rescued` en la celda 3 y volvé a correr.')
    else:
        print(f'No hay candidatos nuevos con threshold={NEW_THRESHOLD}')

## 6. Validación con OCR (Paso 3a)

Después de correr `scripts/fetch_subs.py`, esta celda carga el reporte OCR
y cruza la sincronización sub↔video como evidencia adicional del match.

> Si `sync_ok=False` el sub del YouTube no encaja con la duración del MOV local
> — posible match incorrecto o offset de tiempo.

In [ ]:
if OCR_REPORT.exists():
    ocr = pd.read_csv(OCR_REPORT)
    ok_now = df_edit[df_edit['match'] == 'OK'].copy()
    merged = ok_now.merge(ocr[['yt_id', 'n_segments', 'ocr_confidence',
                                'docx_match_ratio', 'sync_ok']], on='yt_id', how='left')

    suspect = merged[merged['sync_ok'] == False]
    good    = merged[merged['sync_ok'] == True]
    missing = merged[merged['sync_ok'].isna()]

    print(f'sync_ok=True:  {len(good):2d}  — match confirmado por OCR')
    print(f'sync_ok=False: {len(suspect):2d}  — revisar')
    print(f'sin datos OCR: {len(missing):2d}')

    if len(suspect):
        print('\n⚠ Sospechosos (sync_ok=False):')
        display(suspect[['yt_title', 'local_file', 'score',
                          'sync_ok', 'n_segments', 'ocr_confidence']]
                .style.applymap(lambda v: 'background-color: #ffcccc' if v is False else '',
                                subset=['sync_ok']))
else:
    print('⏳ OCR report no disponible aún — correr scripts/fetch_subs.py primero')
    print(f'   (buscando: {OCR_REPORT})')

## 7. Guardar CSV corregido

In [ ]:
# Resumen antes de guardar
counts = df_edit['match'].value_counts()
for k, v in counts.items():
    print(f'  {k:12s}: {v}')
print()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ok_final = df_edit[df_edit['match'] == 'OK']
if len(ok_final):
    axes[0].hist(ok_final['score'], bins=15, color='#2ecc71', edgecolor='white')
    axes[0].axvline(0.35, color='red', linestyle='--', label='Threshold original 0.35')
    axes[0].set_xlabel('Score de matching')
    axes[0].set_ylabel('Videos')
    axes[0].set_title('Distribución de scores (matches OK)')
    axes[0].legend()

cats   = ['OK', 'SIN MATCH', 'RECHAZADO']
vals   = [counts.get(c, 0) for c in cats]
colors = ['#2ecc71', '#e74c3c', '#f39c12']
axes[1].bar(cats, vals, color=colors, edgecolor='white')
axes[1].set_title('Estado del matching')
for i, v in enumerate(vals):
    if v: axes[1].text(i, v + 0.2, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
df_edit.to_csv(CSV_PATH, index=False, encoding='utf-8')
print(f'Guardado: {CSV_PATH}')
print(f'Matches OK finales: {(df_edit["match"] == "OK").sum()}')